> 在线讲义：[第 8 章 · 为什么除以 √d_k](https://wikieden.github.io/transformer-visual/lessons/08-sqrt-dk.html) · 本书《从零读懂 Transformer》

# 第 8 章 · 为什么除以 $\sqrt{d_k}$

配套讲义：[在线版](https://wikieden.github.io/transformer-visual/lessons/08-sqrt-dk.html) · 仓库里 `transformer-deep-dive/notes/lessons/08-sqrt-dk.md`

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的 torch 写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。不需要 GPU。

In [ ]:
import torch

torch.set_printoptions(precision=3, sci_mode=False)  # 打印保留 3 位小数，不用科学计数法
print("torch", torch.__version__)

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一行代码、跑出来是什么。

| 讲义 | 公式 / 说法 | PyTorch 写法 | 跑出来 |
|---|---|---|---|
| 根号 | $\sqrt{64}$ | `math.sqrt(64)` | 8 |
| 抛 d 次硬币，离 0 多远 | 约 $\sqrt d$ | `randint` 造 ±1，`.sum(1).std()` | d = 4、64、512：2.00、7.97、22.79 |
| 随机 q·k | $d_k = 64$ | `(q * k).sum(1).std()` | ≈ 8.0，除以 8 后 ≈ 0.99 |
| 除以 $\sqrt{d_k}$ | [8, −8, 3] ÷ 8 | `torch.softmax(s / 8, -1)` | [0.599, 0.081, 0.320] |
| 除以 $d_k$（太多） | [8, −8, 3] ÷ 64 | `torch.softmax(s / 64, -1)` | [0.37, 0.29, 0.34] |

## 1. 先把要用的东西定义好

这几格只是定义，运行后没有输出也正常。

In [ ]:
import math
import torch

**函数 `coin_flip_spread`**：抛 d 次硬币（每项 ±1）加起来，重复很多次，看总分通常离 0 多远（标准差）。

In [ ]:
def coin_flip_spread(d: int, trials: int = 20000, seed: int = 0) -> float:
    """抛 d 次硬币（每项 ±1）加起来，重复很多次，看总分通常离 0 多远（标准差）。"""
    g = torch.Generator().manual_seed(seed)
    flips = torch.randint(0, 2, (trials, d), generator=g).float() * 2 - 1
    return float(flips.sum(dim=1).std())

**函数 `random_dot_spread`**：q、k 每个数是均值 0、方差 1 的随机数：q·k 通常多大。

In [ ]:
def random_dot_spread(d_k: int, trials: int = 20000, seed: int = 0) -> float:
    """q、k 每个数是均值 0、方差 1 的随机数：q·k 通常多大。"""
    g = torch.Generator().manual_seed(seed)
    q = torch.randn(trials, d_k, generator=g)
    k = torch.randn(trials, d_k, generator=g)
    return float((q * k).sum(dim=1).std())

**函数 `rounded`**：小工具，把结果整理成好读的样子再打印。

In [ ]:
def rounded(t: torch.Tensor, n: int = 3) -> list[float]:
    return [round(float(v), n) for v in t]

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("√64 =", math.sqrt(64), "  torch.sqrt(64) =", float(torch.sqrt(torch.tensor(64.0))))
print("抛硬币：总分通常离 0 多远")
for d in [4, 64, 512]:
    print(f"  d = {d:>3}: 模拟 ≈ {coin_flip_spread(d):5.2f}   √d = {math.sqrt(d):5.2f}")
print(f"随机 q·k（d_k = 64）通常 ≈ ±{random_dot_spread(64):.1f}，除以 8 后 ≈ ±{random_dot_spread(64) / 8:.2f}")
s = torch.tensor([8.0, -8.0, 3.0])

In [ ]:
print("\n分数", s.tolist())
print("  不除    :", rounded(torch.softmax(s, -1)))
print("  除以 8  :", rounded(s / 8), "→", rounded(torch.softmax(s / 8, -1)))
print("  除以 64 :", rounded(s / 64), "→", rounded(torch.softmax(s / 64, -1), 2))

## 3. 代码里才看得到的

1. **用模拟验证 $\sqrt d$**：一次抛 2 万组硬币，看总分通常离 0 多远，结果和 $\sqrt d$ 几乎一样。std = standard deviation（标准差：通常离平均多远，第 19 章细讲）。
2. **`manual_seed`（seed：种子）**：随机数从同一个种子出发，每次跑出来都一样，测试才能核对。
3. **`randn` 造的是均值 0、方差 1 的随机数**：两个这样的 64 维向量做点积，通常就在 ±8 左右，正是论文脚注 4（L198–L201）说的"点积方差 = $d_k$"。

## 4. 自己改一改

- 把上面任意一格里的数字改一改，再按 Shift + Enter，看结果怎么变。

在下面的空格子里写，或者直接改上面的格子。

In [ ]:
# 在这里写你的代码

## 口诀

> 抛 d 次硬币离 0 约 √d，64 个数的点积在 ±8 上下；
> 除以 $\sqrt{d_k}$ 拉回 ±1：不除会饱和，除 $d_k$ 变平均，除根号正好。